# Emptyu 30-Epoch Scale Run

In [ ]:
import os, shutil, subprocess
from pathlib import Path
# Reduce CUDA fragmentation (larger micro-batches / eval on a 16 GB T4) and speed up
# CUDA init (lazy module loading). Must be set before torch is imported; applies to
# every subprocess launched later too.
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('CUDA_MODULE_LOADING', 'LAZY')
REPO = Path('/content/emptyu')
os.chdir('/content')
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/sandeep999-cyber/emptyu.git', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
import torch
assert torch.cuda.is_available(), 'CUDA unavailable'
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CANDIDATES = [Path('/content/drive/MyDrive/storage'), Path('/content/drive/My Computer/storage'), Path('/content/drive/MyDrive/MarketFoundation/storage'), Path('/content/drive/My Computer/MarketFoundation/storage')]
DRIVE_STORAGE = next((p for p in CANDIDATES if p.is_dir()), None)
if DRIVE_STORAGE is None: raise FileNotFoundError('Cannot locate storage on Drive')
DRIVE_BASE = DRIVE_STORAGE.parent
src_training = DRIVE_STORAGE / 'training'
dst_training = REPO / 'storage/training'
MARK = dst_training / '.colab_synced'
if not MARK.exists():
    # Fresh session: copy the canonical (~307 MB) + training data from Drive once.
    # The marker makes re-runs within the same session instant (no re-copy).
    if dst_training.exists(): shutil.rmtree(dst_training)
    shutil.copytree(src_training, dst_training)
    dst_canonical = REPO / 'storage/canonical'
    if dst_canonical.exists(): shutil.rmtree(dst_canonical)
    for symbol in ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']:
        for subdir in ['klines/1m', 'funding', 'open_interest', 'metadata']:
            src = DRIVE_STORAGE / 'canonical/futures' / symbol / subdir
            dst = dst_canonical / 'futures' / symbol / subdir
            if src.is_dir(): shutil.copytree(src, dst)
    MARK.write_text('ok')
for name in ['models', 'logs', 'evaluation']:
    dst, src = REPO / name, DRIVE_BASE / name
    src.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink() if dst.is_symlink() or dst.is_file() else shutil.rmtree(dst)
    dst.symlink_to(src)
import json
fp = json.loads((REPO / 'storage/training/dataset_fingerprint.json').read_text())
assert fp['fingerprint'] == '328a7b67b070b95e47ba450452032a93dfa410431e0cf329de6a4ac7b5ae3875'
assert fp['file_count'] == 510
print('Dataset verified:', fp['fingerprint'][:16] + '...')


In [ ]:
# Dry-run probe: pick micro_batch_size / torch_compile on THIS GPU before the full run.
# Writes a tuned copy to models/foundation/teacher_v1/probe_config.yaml (Drive-backed),
# so later sessions skip the ~5-min probe. Delete that file to re-probe (e.g. new GPU).
PROBE = True  # set False to force-skip
APPLY = True  # auto-write the recommended micro_batch_size/torch_compile
if PROBE:
    import yaml as _yaml
    _applied = REPO / 'models/foundation/teacher_v1/probe_config.yaml'
    if _applied.exists():
        _cfg = _yaml.safe_load(_applied.read_text())['trainer']
        print('Tuned config present (torch_compile=%s, micro_batch=%s); skipping probe.' % (
            _cfg.get('torch_compile'), _cfg.get('micro_batch_size')))
    else:
        env = dict(os.environ, PYTHONPATH=str(REPO))
        cmd = ['python', '-u', '-m', 'src.training.probe']
        if APPLY: cmd += ['--apply']
        subprocess.run(cmd, cwd=REPO, env=env, check=True)


In [ ]:
# After a disconnect, set RESUME_RUN to the Drive-backed run dir to continue training.
RESUME_RUN = None  # None = auto-detect & resume an in-flight run; '' (empty) = force a fresh run

teacher_v1 = REPO / 'models/foundation/teacher_v1'
if RESUME_RUN == '':
    print('RESUME_RUN empty: starting a fresh run (no auto-resume).')
elif RESUME_RUN is None:
    # Auto-detect: if a run exists whose last checkpoint is before the target epoch,
    # resume it instead of silently starting a second fresh run.
    import json as _json
    in_flight = sorted((p for p in teacher_v1.iterdir() if (p / 'latest.json').exists()))[-1] if teacher_v1.exists() and any((p / 'latest.json').exists() for p in teacher_v1.iterdir()) else None
    if in_flight is not None:
        try:
            _cfg = _json.loads((in_flight / 'manifest.json').read_text())['configs']['trainer_config']
            _epochs = _cfg.get('epochs') if isinstance(_cfg, dict) and 'epochs' in _cfg else _cfg['trainer']['epochs']
            _last = _json.loads((in_flight / 'latest.json').read_text())
            _ckp = _last.get('latest') or _last.get('best')
            _cur = int(_ckp.split('epoch')[-1].split('.')[0]) if _ckp else 0
        except Exception:
            _cur, _epochs = 0, None
        if _epochs and _cur < _epochs:
            RESUME_RUN = str(in_flight.relative_to(REPO))
            print('Auto-resuming in-flight run %s at epoch %s/%s.' % (RESUME_RUN, _cur, _epochs))
            print('Set RESUME_RUN = "" to start a fresh run instead.')
        else:
            print('No incomplete run found; starting a fresh run.')

# Prefer the probe-tuned config (Drive-backed) when present; otherwise use the default scale30 config.
trainer_cfg = REPO / 'configs/trainer_v1_scale30.yaml'
probe_cfg = REPO / 'models/foundation/teacher_v1/probe_config.yaml'
if probe_cfg.exists():
    trainer_cfg = probe_cfg
    print('Using tuned config:', trainer_cfg)
cmd = ['python', '-u', '-m', 'src.training.train_teacher', '--model-config', 'configs/model_v1.yaml', '--optimizer-config', 'configs/optimizer_v1.yaml', '--trainer-config', str(trainer_cfg)]
if RESUME_RUN: cmd += ['--resume', RESUME_RUN]
# Stream training output live; on failure the last ~40 lines are printed inline.
_proc = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
_lines = []
for _line in _proc.stdout:
    print(_line, end='', flush=True)
    _lines.append(_line)
_proc.wait()
if _proc.returncode != 0:
    print('===== TRAINING FAILED (exit %s) — last output =====' % _proc.returncode)
    print(''.join(_lines[-40:]))
    print('=====================================================')
    raise RuntimeError('Training failed with exit code %s. See output above.' % _proc.returncode)

runs = sorted(p for p in teacher_v1.iterdir() if (p / 'manifest.json').exists())
if not runs: raise FileNotFoundError('No completed checkpoint found')
os.environ['CHECKPOINT_DIR'] = str(runs[-1])
print('CHECKPOINT_DIR:', os.environ['CHECKPOINT_DIR'])


In [ ]:
env = dict(os.environ, PYTHONPATH=str(REPO))
# Mean-pooling only: the scale-run protocol compares checkpoints on the mean-pool
# linear probe; all-pooling analysis is already covered by the 10-epoch pilot.
subprocess.run(['python', '-u', '-m', 'src.evaluation.embedding.linear_probe', '--checkpoint', os.environ['CHECKPOINT_DIR'], '--pooling', 'mean', '--batch-size', '64'], cwd=REPO, env=env, check=True)
subprocess.run(['python', '-u', '-m', 'src.evaluation.baselines.runner', '--checkpoint', os.environ['CHECKPOINT_DIR'], '--pooling', 'mean', '--max-windows', '1500', '--batch-size', '64', '--out', 'evaluation/baselines'], cwd=REPO, env=env, check=True)


In [ ]:
archive_dest = DRIVE_BASE / 'phase2_results'
shutil.make_archive(str(archive_dest), 'zip', str(REPO / 'evaluation'))
for name in ['index.duckdb', 'experiment_registry.duckdb']:
    local, remote = REPO / 'storage/training' / name, DRIVE_STORAGE / 'training' / name
    if local.exists(): shutil.copy2(local, remote)
print('Archived:', str(archive_dest) + '.zip')
